# Step 06 — the figure

Patients × proteins, both axes grouped, so an endotype reads as a **block** rather than as two
dendrograms glued to the sides of a heatmap.

In [ ]:
suppressMessages({library(ComplexHeatmap); library(circlize)})
w <- readRDS("artifacts/wgcna_A.rds"); ee <- readRDS("artifacts/eigengenes_A.rds")
sig <- readRDS("artifacts/sig_modules_A.rds"); en <- readRDS("artifacts/endotypes_A.rds")
X <- w$X; m <- w$meta; mods <- w$mods; endo <- en$endo

## Three choices that decide what the picture shows

**Cap each module's contribution.** Twelve proteins per module, ranked by module membership. Without
a cap, turquoise's 2,419 proteins would occupy the whole width and a 12-protein interferon module
would be invisible.

**Scale per protein, across patients. One axis only.** Z-scoring patients as well forces each row to
sum to zero and manufactures anticorrelation between proteins.

**ward.D2 within each block.** Ward minimises the increase in within-cluster variance at each merge,
which produces tight rectangles rather than the chained strands single-linkage gives.

In [ ]:
sel <- unlist(lapply(sig, function(md0) {
  g <- colnames(X)[mods == md0]
  g[order(-abs(cor(X[, g, drop = FALSE], ee$ME[[paste0("ME", md0)]])))][1:min(12, length(g))]
}))
ord  <- sig[order(sapply(sig, function(k) sum(mods == k)))]   # small modules first
modf <- factor(mods[match(sel, colnames(X))], levels = ord)
Z    <- scale(as.matrix(X[, sel]))
sprintf("%d modules, %d proteins", length(sig), length(sel))

In [ ]:
options(repr.plot.width = 22, repr.plot.height = 13)
ann <- m[, c("Disease_activity","SLEDAI_2K","C3_level","Ro_52_status","La_status","Duration_years")]
draw(Heatmap(Z, name = "z-score",
  col = colorRamp2(c(-2, 0, 2), c("#2166AC", "white", "#B2182B")),
  row_split = endo, column_split = modf,
  cluster_rows = TRUE, cluster_row_slices = FALSE, cluster_columns = TRUE,
  clustering_method_rows = "ward.D2", clustering_method_columns = "ward.D2",
  show_row_dend = TRUE, row_dend_width = unit(25, "mm"), show_row_names = FALSE,
  column_names_side = "top", column_names_gp = gpar(fontsize = 6), column_dend_side = "top",
  column_title_gp = gpar(fontsize = 7), column_title_rot = 90,
  left_annotation = rowAnnotation(df = ann, annotation_name_gp = gpar(fontsize = 7)),
  row_title = "endotype"))

## How to read it

Find the **royalblue** column block. It is narrow — twelve proteins — and it is the one block whose
high rows are a scattered subset cutting across *both* endotypes rather than filling one of them.
Those are the interferon-high patients. They are a candidate endotype that the step-05 partition
does not isolate, because the eigenprotein k-means is dominated by turquoise and blue.

The obvious next analysis is to cluster patients on **royalblue alone** and ask whether that group
is coherent and whether it replicates in cohorts B and C.